In [ ]:
import os
import numpy as np
import h5py
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
import networkx as nx
import pandas as pd
from scipy import ndimage
from collections import Counter

TISSUE_NAMES = [
    "Tumor", "Empty", "Fibrous", "Inflammation",
    "Necrosis", "Normal", "Reactive", "Steatosis"
]

TISSUE_COLORS = [
    "#E64B35", "#FFFFFF", "#7E57C2", "#6E6E6E",
    "#F06292", "#66BB6A", "#26A69A", "#F39C12"
]

In [ ]:
def load_segmap(segmap_npy):
    seg_prob = np.load(segmap_npy)
    seg_label = np.argmax(seg_prob, axis=2)
    return seg_prob, seg_label

def load_coords_from_h5(h5_path, coord_key="coords"):
    with h5py.File(h5_path, "r") as f:
        coords = f[coord_key][:]
    return coords.astype(np.int32)

def get_foreground_bbox_clean(seg_label, empty_label=1, min_area=500):

    fg_mask = seg_label != empty_label

    labeled, num = ndimage.label(fg_mask)
    if num == 0:
        raise ValueError("Foreground mask is empty.")

    sizes = np.bincount(labeled.ravel())
    sizes[0] = 0

    keep_labels = np.where(sizes >= min_area)[0]

    if len(keep_labels) == 0:
        fg_clean = fg_mask
    else:
        fg_clean = np.isin(labeled, keep_labels)

    ys, xs = np.where(fg_clean)
    if len(xs) == 0 or len(ys) == 0:
        raise ValueError("Clean foreground mask is empty.")

    return xs.min(), xs.max(), ys.min(), ys.max(), fg_clean

In [ ]:
def coords_to_seg_boxes_bbox_sample_scale(
    coords,
    seg_label,
    patch_size=512,
    empty_label=1,
):
    coords = coords.astype(np.float64)

    # WSI / CLAM coords bbox
    x_min_wsi = coords[:, 0].min()
    y_min_wsi = coords[:, 1].min()
    x_max_wsi = coords[:, 0].max()
    y_max_wsi = coords[:, 1].max()

    wsi_w = (x_max_wsi - x_min_wsi) + patch_size
    wsi_h = (y_max_wsi - y_min_wsi) + patch_size

    x_min_seg, x_max_seg, y_min_seg, y_max_seg, fg_mask = get_foreground_bbox_clean(seg_label, empty_label=empty_label, min_area=500)

    seg_w = x_max_seg - x_min_seg + 1
    seg_h = y_max_seg - y_min_seg + 1

    scale_x = seg_w / wsi_w
    scale_y = seg_h / wsi_h

    x0 = coords[:, 0] - x_min_wsi
    y0 = coords[:, 1] - y_min_wsi

    x1 = x_min_seg + x0 * scale_x
    y1 = y_min_seg + y0 * scale_y
    x2 = x_min_seg + (x0 + patch_size) * scale_x
    y2 = y_min_seg + (y0 + patch_size) * scale_y

    ix = ((x1 + x2) / 2).astype(int)
    iy = ((y1 + y2) / 2).astype(int)

    Hm, Wm = seg_label.shape
    ix = np.clip(ix, 0, Wm - 1)
    iy = np.clip(iy, 0, Hm - 1)

    mapping_info = {
        "x_min_wsi": float(x_min_wsi),
        "y_min_wsi": float(y_min_wsi),
        "x_max_wsi": float(x_max_wsi),
        "y_max_wsi": float(y_max_wsi),
        "wsi_w": float(wsi_w),
        "wsi_h": float(wsi_h),
        "x_min_seg": int(x_min_seg),
        "x_max_seg": int(x_max_seg),
        "y_min_seg": int(y_min_seg),
        "y_max_seg": int(y_max_seg),
        "seg_w": int(seg_w),
        "seg_h": int(seg_h),
        "scale_x": float(scale_x),
        "scale_y": float(scale_y),
        "scale_ratio_x_over_y": float(scale_x / scale_y),
    }

    return iy, ix, x1, y1, x2, y2, fg_mask, mapping_info

In [ ]:
def extract_sub_nodes_from_pgexplainer_result(pgexplainer_result_file, sample_idx=1):
    test_explanations = torch.load(pgexplainer_result_file, map_location="cpu")
    exp = test_explanations[sample_idx]

    top_edge_index = exp["top_edge_index"]
    if torch.is_tensor(top_edge_index):
        top_edge_index = top_edge_index.detach().cpu().numpy()

    sub_nodes = np.unique(top_edge_index.reshape(-1))
    return sub_nodes, exp


def filter_small_components(top_edge_index, min_size=5):
    edges = top_edge_index.T
    G = nx.Graph()
    G.add_edges_from(edges)

    components = list(nx.connected_components(G))

    keep_nodes = set()
    for comp in components:
        if len(comp) >= min_size:
            keep_nodes.update(comp)

    new_edges = [
        (u, v) for (u, v) in edges
        if u in keep_nodes and v in keep_nodes
    ]

    if len(new_edges) == 0:
        return np.empty((2, 0), dtype=int), np.array([], dtype=int)

    new_edge_index = np.array(new_edges).T
    sub_nodes = np.array(sorted(list(keep_nodes)))

    return new_edge_index, sub_nodes


def match_top_edges_with_weights(edge_index, edge_mask, top_edge_index):

    edge_index_np = edge_index.T  # (E, 2)
    top_edges_np = top_edge_index.T  # (K, 2)

    matched_weights = []

    for e in top_edges_np:
        matches = np.where(
            (edge_index_np[:, 0] == e[0]) &
            (edge_index_np[:, 1] == e[1])
        )[0]

        if len(matches) > 0:
            matched_weights.append(edge_mask[matches[0]])
        else:
            matched_weights.append(0.0)

    return top_edge_index, np.array(matched_weights)


def pool_patch_tissue_from_boxes_foreground_only(
    seg_prob,
    x1, y1, x2, y2,
    empty_label=1,
    min_fg_ratio=0.05
):
    Hm, Wm, C = seg_prob.shape
    N = len(x1)

    patch_tissue_prob = np.zeros((N, C), dtype=np.float32)
    patch_tissue_label = np.full(N, empty_label, dtype=np.int64)
    patch_fg_ratio = np.zeros(N, dtype=np.float32)

    seg_label = np.argmax(seg_prob, axis=2)

    for i in range(N):
        xa = int(np.floor(x1[i]))
        ya = int(np.floor(y1[i]))
        xb = int(np.ceil(x2[i]))
        yb = int(np.ceil(y2[i]))

        xa = np.clip(xa, 0, Wm - 1)
        xb = np.clip(xb, xa + 1, Wm)
        ya = np.clip(ya, 0, Hm - 1)
        yb = np.clip(yb, ya + 1, Hm)

        region_prob = seg_prob[ya:yb, xa:xb, :]
        region_label = seg_label[ya:yb, xa:xb]

        if region_prob.size == 0:
            continue

        fg_mask = region_label != empty_label
        patch_fg_ratio[i] = fg_mask.mean()

        if patch_fg_ratio[i] < min_fg_ratio:
            patch_tissue_prob[i, empty_label] = 1.0
            patch_tissue_label[i] = empty_label
            continue

        fg_prob = region_prob[fg_mask]
        mean_prob = fg_prob.mean(axis=0)

        mean_prob[empty_label] = 0.0
        s = mean_prob.sum()
        if s > 0:
            mean_prob = mean_prob / s

        patch_tissue_prob[i] = mean_prob
        patch_tissue_label[i] = int(np.argmax(mean_prob))

    return patch_tissue_prob, patch_tissue_label, patch_fg_ratio


def summarize_subgraph_tissue_pixels_from_boxes(
        seg_label,
        sub_nodes,
        x1, y1, x2, y2,
        tissue_names=TISSUE_NAMES,
        ignore_empty=True,
        empty_label=1,
):
    counts = np.zeros(len(tissue_names), dtype=np.int64)

    for node in sub_nodes:
        node = int(node)

        patch_labels = seg_label[
            int(y1[node]):int(y2[node]),
            int(x1[node]):int(x2[node])
        ]

        if patch_labels.size == 0:
            continue

        if ignore_empty:
            patch_labels = patch_labels[patch_labels != empty_label]

        if patch_labels.size == 0:
            continue

        counts += np.bincount(
            patch_labels.ravel(),
            minlength=len(tissue_names)
        )

    total = counts.sum()
    ratios = counts / total if total > 0 else np.full(len(tissue_names), np.nan)

    summary = {
        "subgraph_total_tissue_pixels": int(total)
    }

    for i, name in enumerate(tissue_names):
        summary[f"subgraph_{name}_pixel_count"] = int(counts[i])
        summary[f"subgraph_{name}_pixel_ratio"] = float(ratios[i])

    return summary


def summarize_subgraph_tissue(sub_tissue_label, sub_tissue_prob=None):
    counts = np.bincount(sub_tissue_label, minlength=8)
    ratios = counts / counts.sum() if counts.sum() > 0 else counts.astype(float)

    print("=== Subgraph tissue ratio (argmax labels) ===")
    for i, name in enumerate(TISSUE_NAMES):
        print(f"{i:>1} {name:<14}: count={counts[i]:>4d}, ratio={ratios[i]:.4f}")

    result = {"counts": counts, "ratios": ratios}

    if sub_tissue_prob is not None:
        mean_prob = sub_tissue_prob.mean(axis=0)
        print("\n=== Subgraph tissue composition (mean probabilities) ===")
        for i, name in enumerate(TISSUE_NAMES):
            print(f"{i:>1} {name:<14}: mean_prob={mean_prob[i]:.4f}")
        result["mean_prob"] = mean_prob

    return result

def summarize_all_edge_tissue_types(
        edge_index,
        patch_tissue_label,
        edge_weights=None,
        tissue_names=TISSUE_NAMES,
        ignore_empty=True,
        empty_label=1,
):
    edge_index = np.asarray(edge_index)
    patch_tissue_label = np.asarray(patch_tissue_label)

    if edge_weights is None:
        edge_weights = np.ones(edge_index.shape[1], dtype=float)
    else:
        edge_weights = np.asarray(edge_weights)

    edge_count = Counter()
    edge_weight = Counter()

    total_edges = 0
    total_weight = 0.0

    for e in range(edge_index.shape[1]):
        u, v = edge_index[:, e]
        u, v = int(u), int(v)

        tu = int(patch_tissue_label[u])
        tv = int(patch_tissue_label[v])
        w = float(edge_weights[e])

        if ignore_empty and (tu == empty_label or tv == empty_label):
            continue

        a, b = sorted([tu, tv])
        edge_type = f"{tissue_names[a]}__{tissue_names[b]}"

        edge_count[edge_type] += 1
        edge_weight[edge_type] += w

        total_edges += 1
        total_weight += w

    summary = {
        "total_tissue_edges": total_edges,
        "total_tissue_edge_weight": total_weight,
    }

    valid_labels = [i for i in range(len(tissue_names))]
    if ignore_empty:
        valid_labels = [i for i in valid_labels if i != empty_label]

    for idx_i, i in enumerate(valid_labels):
        for j in valid_labels[idx_i:]:
            edge_type = f"{tissue_names[i]}__{tissue_names[j]}"

            count = edge_count.get(edge_type, 0)
            weight = edge_weight.get(edge_type, 0.0)

            summary[f"{edge_type}_edge_count"] = count
            summary[f"{edge_type}_edge_ratio"] = (
                count / total_edges if total_edges > 0 else np.nan
            )
            summary[f"{edge_type}_edge_weight_sum"] = weight
            summary[f"{edge_type}_edge_weight_ratio"] = (
                weight / total_weight if total_weight > 0 else np.nan
            )

    print("Non-zero edge types:", {k: v for k, v in edge_count.items() if v > 0})
    print("total_edges:", total_edges)

    return summary


def summarize_segmap_tissue_pixels(
        seg_label,
        tissue_names=TISSUE_NAMES,
        ignore_empty=True,
        empty_label=1,
):
    seg_label = np.asarray(seg_label)

    if ignore_empty:
        valid_mask = seg_label != empty_label
    else:
        valid_mask = np.ones_like(seg_label, dtype=bool)

    valid_labels = seg_label[valid_mask]

    counts = np.bincount(
        valid_labels.ravel(),
        minlength=len(tissue_names)
    )

    total = counts.sum()

    ratios = counts / total if total > 0 else np.full(len(tissue_names), np.nan)

    summary = {"wsi_total_tissue_pixels": int(total)}

    for i, name in enumerate(tissue_names):
        summary[f"wsi_{name}_pixel_count"] = int(counts[i])
        summary[f"wsi_{name}_pixel_ratio"] = float(ratios[i])

    return summary


def plot_overlay(seg_label, ix, iy, top_edge_index, top_weights, title="", save_path=None):
    cmap = ListedColormap(TISSUE_COLORS)

    plt.figure(figsize=(8, 8))
    plt.imshow(seg_label, cmap=cmap, interpolation="nearest", alpha=0.9)

    w = top_weights.astype(float).copy()

    order = np.argsort(w)
    rank = np.empty_like(order, dtype=float)
    rank[order] = np.linspace(0, 1, len(w))

    for i in range(top_edge_index.shape[1]):
        u = top_edge_index[0, i]
        v = top_edge_index[1, i]

        x1, y1 = ix[u], iy[u]
        x2, y2 = ix[v], iy[v]

        r = rank[i]

        gray = 0.75 - 0.3 * r

        alpha = 0.4 + 0.5 * r
        linewidth = 0.4 + 0.7 * r

        plt.plot(
            [x1, x2],
            [y1, y2],
            color=(gray, gray, gray),
            alpha=alpha,
            linewidth=linewidth,
            zorder=2
        )

    sub_nodes = np.unique(top_edge_index.reshape(-1))

    plt.scatter(ix[sub_nodes], iy[sub_nodes], s=40, c="white", alpha=0.6, linewidths=0,zorder=3)

    plt.scatter(ix[sub_nodes], iy[sub_nodes], s=15, c="#00BCD4", edgecolors="#006064",linewidths=0.8, zorder=4)

    patches = [mpatches.Patch(facecolor=TISSUE_COLORS[i], edgecolor='#333333', linewidth=0.6, label=f"{i}: {TISSUE_NAMES[i]}") for i in range(8)]
    patches.append(plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#00BCD4', markeredgecolor='#006064', markersize=8, linestyle='None', label='PGExplainer subgraph'))
    plt.legend(handles=patches, bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

    plt.title(title)
    plt.axis("off")
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


def overlay_pgexplainer_with_pathfinder_refined(
        pgexplainer_result_file,
        sample_idx,
        segmap_npy,
        h5_path,
        patch_size=1024,
        save_overlay_path=None,
        patient_id=None,
):
    sub_nodes, exp = extract_sub_nodes_from_pgexplainer_result(pgexplainer_result_file=pgexplainer_result_file, sample_idx=sample_idx)

    print(f"Sample index: {sample_idx}")

    edge_index = exp["edge_index"].detach().cpu().numpy()
    edge_mask = exp["edge_mask"].detach().cpu().numpy()
    top_edge_index = exp["top_edge_index"].detach().cpu().numpy()

    top_edge_index_clean, sub_nodes_clean = filter_small_components(top_edge_index, min_size=6)
    print(f"Number of subgraph nodes: {len(sub_nodes_clean)}")

    top_edge_index, top_weights = match_top_edges_with_weights(edge_index, edge_mask, top_edge_index_clean)

    seg_prob, seg_label = load_segmap(segmap_npy)
    wsi_tissue_pixel_summary = summarize_segmap_tissue_pixels(
        seg_label=seg_label,
        tissue_names=TISSUE_NAMES,
        ignore_empty=True,
        empty_label=1,
    )

    coords = load_coords_from_h5(h5_path)

    iy, ix, x1, y1, x2, y2, fg_mask, mapping_info = coords_to_seg_boxes_bbox_sample_scale(
        coords=coords,
        seg_label=seg_label,
        patch_size=patch_size,
        empty_label=1
    )

    print(
        f"Sample-scale mapping: "
        f"scale_x={mapping_info['scale_x']:.6f}, "
        f"scale_y={mapping_info['scale_y']:.6f}, "
        f"scale_ratio={mapping_info['scale_ratio_x_over_y']:.4f}"
    )

    patch_tissue_prob, patch_tissue_label, patch_fg_ratio = pool_patch_tissue_from_boxes_foreground_only(
        seg_prob=seg_prob,
        x1=x1, y1=y1, x2=x2, y2=y2,
        empty_label=1,
        min_fg_ratio=0.05
    )

    subgraph_tissue_pixel_summary = summarize_subgraph_tissue_pixels_from_boxes(
        seg_label=seg_label,
        sub_nodes=sub_nodes_clean,
        x1=x1, y1=y1, x2=x2, y2=y2,
        tissue_names=TISSUE_NAMES,
        ignore_empty=True,
        empty_label=1,
    )

    sub_tissue_label = patch_tissue_label[sub_nodes_clean]
    sub_tissue_prob = patch_tissue_prob[sub_nodes_clean]
    sub_fg_ratio = patch_fg_ratio[sub_nodes_clean]

    print("unique sub tissue:", np.unique(sub_tissue_label, return_counts=True))

    u_nodes = np.unique(top_edge_index)
    edge_node_labels = patch_tissue_label[u_nodes]
    print("unique edge-node tissue:", np.unique(edge_node_labels, return_counts=True))
    print("number of unique nodes in edges:", len(u_nodes))
    print("number of sub_nodes_clean:", len(sub_nodes_clean))

    summary = summarize_subgraph_tissue(sub_tissue_label, sub_tissue_prob)

    edge_tissue_summary = summarize_all_edge_tissue_types(
        edge_index=top_edge_index,
        patch_tissue_label=patch_tissue_label,
        edge_weights=top_weights,
        ignore_empty=True,
        empty_label=1)

    print("Subgraph mean foreground ratio:", sub_fg_ratio.mean())
    print("Subgraph low-foreground patches:", (sub_fg_ratio < 0.05).sum(), "/", len(sub_fg_ratio))

    plot_overlay(
        seg_label=seg_label,
        ix=ix,
        iy=iy,
        top_edge_index=top_edge_index,
        top_weights=top_weights,
        title="Subgraph explanation over tissue segmentation map",
        save_path=save_overlay_path
    )

    return {
        "patient_id": patient_id,
        "sample_idx": sample_idx,
        "exp": exp,
        "sub_nodes": sub_nodes_clean,
        "coords": coords,
        "seg_prob": seg_prob,
        "seg_label": seg_label,
        "ix": ix,
        "iy": iy,
        "patch_tissue_label": patch_tissue_label,
        "patch_tissue_prob": patch_tissue_prob,
        "sub_tissue_label": sub_tissue_label,
        "sub_tissue_prob": sub_tissue_prob,
        "summary": summary,
        "edge_tissue_summary": edge_tissue_summary,
        "subgraph_tissue_pixel_summary": subgraph_tissue_pixel_summary,
        "wsi_tissue_pixel_summary": wsi_tissue_pixel_summary,
        "mapping_info": mapping_info,
        "patch_fg_ratio": patch_fg_ratio,
        "sub_fg_ratio": sub_fg_ratio,
    }

In [ ]:
def save_pgexplainer_tissue_summary_csv(all_results, save_csv_path, tissue_names=None):
    if tissue_names is None:
        tissue_names = TISSUE_NAMES

    rows = []

    for res in all_results:
        summary = res["summary"]

        counts = summary["counts"]
        ratios = summary["ratios"]
        mean_prob = summary.get("mean_prob", np.full(len(tissue_names), np.nan))

        row = {
            "patient_id": res.get("patient_id", None),
            "sample_idx": res.get("sample_idx", None),
            "num_subgraph_nodes": len(res.get("sub_nodes", [])),
        }

        # counts
        for i, name in enumerate(tissue_names):
            row[f"{name}_count"] = int(counts[i])

        # ratios
        for i, name in enumerate(tissue_names):
            row[f"{name}_ratio"] = float(ratios[i])

        # mean probabilities
        for i, name in enumerate(tissue_names):
            row[f"{name}_mean_prob"] = float(mean_prob[i])

        # edge tissue interaction summary
        edge_summary = res.get("edge_tissue_summary", {})

        # subgraph tissue pixel summary
        sub_pixel_summary = res.get("subgraph_tissue_pixel_summary", {})
        for k, v in sub_pixel_summary.items():
            row[k] = v

        # WSI tissue pixel summary
        wsi_summary = res.get("wsi_tissue_pixel_summary", {})
        for k, v in wsi_summary.items():
            row[k] = v

        for k, v in edge_summary.items():
            if isinstance(v, (np.integer, int)):
                row[k] = int(v)
            elif isinstance(v, (np.floating, float)):
                row[k] = float(v)
            else:
                row[k] = v

        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(save_csv_path, index=False)

    print(f"[Saved CSV] {save_csv_path}")
    print(df.head())

    return df

In [ ]:
def run_overlay_for_testset(
    test_ids,
    segmap_dir,
    h5_dir,
    img_save_dir,
    pgexplainer_result_file,
    patch_size=1024
):
    os.makedirs(img_save_dir, exist_ok=True)

    pg_results = torch.load(pgexplainer_result_file, map_location="cpu")

    id2idx = {}
    for i, exp in enumerate(pg_results):
        pid = exp.get("patient_id", None)
        if pid is not None:
            id2idx[pid] = i

    print(f"Input IDs: {len(test_ids)}")
    print(f"Explanation IDs: {len(id2idx)}")

    all_results = []
    fail_log = []

    for pid in test_ids:

        if pid not in id2idx:
            print(f"[Skip] explanation not found: {pid}")
            fail_log.append({
                "patient_id": pid,
                "reason": "explanation_not_found"
            })
            continue

        sample_idx = id2idx[pid]
        slide_id = f"{pid}-01Z-00-DX1"

        segmap_npy = os.path.join(segmap_dir, "40x-" + slide_id + ".npy")
        h5_path = os.path.join(h5_dir, slide_id + ".h5")
        img_save_path = os.path.join(img_save_dir, slide_id + "_overlay.png")

        if not os.path.exists(segmap_npy):
            print(f"[Skip] segmap not found: {pid}")
            fail_log.append({
                "patient_id": pid,
                "reason": "segmap_not_found",
                "path": segmap_npy
            })
            continue

        if not os.path.exists(h5_path):
            print(f"[Skip] h5 not found: {pid}")
            fail_log.append({
                "patient_id": pid,
                "reason": "h5_not_found",
                "path": h5_path
            })
            continue

        print(f"[Processing] pid={pid}, sample_idx={sample_idx}")

        try:
            res = overlay_pgexplainer_with_pathfinder_refined(
                pgexplainer_result_file=pgexplainer_result_file,
                sample_idx=sample_idx,
                segmap_npy=segmap_npy,
                h5_path=h5_path,
                patch_size=patch_size,
                save_overlay_path=img_save_path,
                patient_id=pid,
            )

            res["patient_id"] = pid
            res["sample_idx"] = sample_idx
            all_results.append(res)

        except Exception as e:
            print(f"[Error] {pid}: {e}")
            fail_log.append({
                "patient_id": pid,
                "reason": "overlay_error",
                "error": str(e)
            })
            continue

    print("All done!")
    print("success:", len(all_results))
    print("failed:", len(fail_log))

    if len(fail_log) > 0:
        fail_df = pd.DataFrame(fail_log)
        display(fail_df)

    return all_results, fail_log

In [ ]:
# CombinedModel
# pgexplainer_result_file = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/CombinedModel_ss/train_explanations.pt"
# PatchGCN
# pgexplainer_result_file = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/PatchGCN_spatial_check/train_explanations.pt"
# H2GCN
pgexplainer_result_file = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/H2GCN_spatial_check/train_explanations.pt"

splits_dir = '/root/Desktop/data/private/LIHC/5foldcv/lihc_343/splits_2.csv'
df = pd.read_csv(splits_dir)
train_ids = (df["train"].dropna().tolist() + df["validation"].dropna().tolist())

segmap_dir = "/root/Desktop/data/private/PagSegNet/seg_hjx/resnext50_32x4d/"
h5_dir = "/root/Desktop/data/private/LIHC/feature_DX_UNI2/h5_files/"

# CombinedModel
# img_save_dir = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/CombinedModel_ss/vis_overlay/"
# save_csv_path = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/CombinedModel_ss/train_tissue_summary.csv"
# PatchGCN
# img_save_dir = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/PatchGCN_spatial_check/vis_overlay/"
# save_csv_path = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/PatchGCN_spatial_check/train_tissue_summary.csv"
# H2GCN
img_save_dir = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/H2GCN_spatial_check/vis_overlay/"
save_csv_path = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/H2GCN_spatial_check/train_tissue_summary.csv"

all_results, fail_log = run_overlay_for_testset(
    test_ids=train_ids,
    segmap_dir=segmap_dir,
    h5_dir=h5_dir,
    img_save_dir=img_save_dir,
    pgexplainer_result_file=pgexplainer_result_file,
    patch_size=1024
)

df_tissue = save_pgexplainer_tissue_summary_csv(
    all_results=all_results,
    save_csv_path=save_csv_path
)

In [ ]:
# CombinedModel
# pgexplainer_result_file = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/CombinedModel_ss/test_explanations.pt"
# PatchGCN
# pgexplainer_result_file = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/PatchGCN_spatial_check/test_explanations.pt"
# H2GCN
pgexplainer_result_file = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/H2GCN_spatial_check/test_explanations.pt"

splits_dir = '/root/Desktop/data/private/LIHC/5foldcv/lihc_343/splits_2.csv'
df = pd.read_csv(splits_dir)
test_ids = df["test"].dropna().tolist()

segmap_dir = "/root/Desktop/data/private/PagSegNet/seg_hjx/resnext50_32x4d/"
h5_dir = "/root/Desktop/data/private/LIHC/feature_DX_UNI2/h5_files/"

# CombinedModel
# img_save_dir = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/CombinedModel_ss/vis_overlay/"
# save_csv_path = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/CombinedModel_ss/test_tissue_summary.csv"
# PatchGCN
# img_save_dir = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/PatchGCN_spatial_check/vis_overlay/"
# save_csv_path = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/PatchGCN_spatial_check/test_tissue_summary.csv"
# H2GCN
img_save_dir = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/H2GCN_spatial_check/vis_overlay/"
save_csv_path = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/H2GCN_spatial_check/test_tissue_summary.csv"

all_results, fail_log = run_overlay_for_testset(
    test_ids=test_ids,
    segmap_dir=segmap_dir,
    h5_dir=h5_dir,
    img_save_dir=img_save_dir,
    pgexplainer_result_file=pgexplainer_result_file,
    patch_size=1024
)

df_tissue = save_pgexplainer_tissue_summary_csv(
    all_results=all_results,
    save_csv_path=save_csv_path
)